<a href="https://colab.research.google.com/github/sinhaanshuk1712-alt/demographic-shift-econometrics/blob/main/world_project_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wbgapi statsmodels
import wbgapi as wb
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# 1. Define Target Regions: Asian Tigers (KOR, SGP) vs. Emerging South Asia (IND, BGD)
countries = ['KOR', 'SGP', 'IND', 'BGD']

# 2. Extract Data from World Bank API
df_pop = wb.data.DataFrame('SP.POP.1564.TO.ZS', countries, time=range(1990, 2025), numericTimeKeys=True)
df_gdp = wb.data.DataFrame('NY.GDP.PCAP.KD', countries, time=range(1990, 2025), numericTimeKeys=True)

# 3. Reshape to "Long Format" (Crucial for Panel Data Econometrics)
df_pop_long = df_pop.reset_index().melt(id_vars='economy', var_name='Year', value_name='Working_Age_Pct')
df_gdp_long = df_gdp.reset_index().melt(id_vars='economy', var_name='Year', value_name='GDP_Per_Capita')

# 4. Merge datasets on both Economy and Year
df_panel = pd.merge(df_pop_long, df_gdp_long, on=['economy', 'Year'])
df_panel.dropna(inplace=True)

# 5. Transform Dependent Variable to measure percentage growth
df_panel['Log_GDP'] = np.log(df_panel['GDP_Per_Capita'])

# 6. Two-Way Fixed Effects Panel Regression
# C(economy) controls for Country Fixed Effects (unobserved country-specific traits)
# C(Year) controls for Time Fixed Effects (global macroeconomic shocks)
model = smf.ols('Log_GDP ~ Working_Age_Pct + C(economy) + C(Year)', data=df_panel).fit()

print("=== Two-Way Fixed Effects Panel Regression Results ===")
print(model.summary())

=== Two-Way Fixed Effects Panel Regression Results ===
                            OLS Regression Results                            
Dep. Variable:                Log_GDP   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.998
Method:                 Least Squares   F-statistic:                     2404.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):          6.94e-134
Time:                        18:11:56   Log-Likelihood:                 196.26
No. Observations:                 140   AIC:                            -314.5
Df Residuals:                     101   BIC:                            -199.8
Df Model:                          38                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------